In [ ]:
#%pip install torch torchvision

In [ ]:
import torch.utils.data as utils
import torch.nn.functional as F
import torch
import torch.nn as nn
from torch.autograd import Variable
from torch.nn.parameter import Parameter
import numpy as np
import pandas as pd
import math
import time

In [ ]:
# import matplotlib.pyplot as plt
# %matplotlib inline

In [ ]:
print(torch.__version__)

2.10.0+cu128


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
def PrepareDataset(speed_matrix, BATCH_SIZE = 40, seq_len = 10, pred_len = 1, train_propotion = 0.7, valid_propotion = 0.2):
    """ Prepare training and testing datasets and dataloaders.

    Convert speed/volume/occupancy matrix to training and testing dataset.
    The vertical axis of speed_matrix is the time axis and the horizontal axis
    is the spatial axis.

    Args:
        speed_matrix: a Matrix containing spatial-temporal speed data for a network
        seq_len: length of input sequence
        pred_len: length of predicted sequence
    Returns:
        Training dataloader
        Testing dataloader
    """
    time_len = speed_matrix.shape[0]

    max_speed = speed_matrix.max().max()
    speed_matrix =  speed_matrix / max_speed

    speed_sequences, speed_labels = [], []
    for i in range(time_len - seq_len - pred_len):
        speed_sequences.append(speed_matrix.iloc[i:i+seq_len].values)
        speed_labels.append(speed_matrix.iloc[i+seq_len:i+seq_len+pred_len].values)
    speed_sequences, speed_labels = np.asarray(speed_sequences), np.asarray(speed_labels)

    # shuffle and split the dataset to training and testing datasets
    sample_size = speed_sequences.shape[0]
    index = np.arange(sample_size, dtype = int)
    np.random.shuffle(index) # не используем далее - в итоге не шафлим

    train_index = int(np.floor(sample_size * train_propotion))
    valid_index = int(np.floor(sample_size * ( train_propotion + valid_propotion)))

    train_data, train_label = speed_sequences[:train_index], speed_labels[:train_index]
    valid_data, valid_label = speed_sequences[train_index:valid_index], speed_labels[train_index:valid_index]
    test_data, test_label = speed_sequences[valid_index:], speed_labels[valid_index:]

    train_data, train_label = torch.Tensor(train_data), torch.Tensor(train_label)
    valid_data, valid_label = torch.Tensor(valid_data), torch.Tensor(valid_label)
    test_data, test_label = torch.Tensor(test_data), torch.Tensor(test_label)

    train_dataset = utils.TensorDataset(train_data, train_label)
    valid_dataset = utils.TensorDataset(valid_data, valid_label)
    test_dataset = utils.TensorDataset(test_data, test_label)

    train_dataloader = utils.DataLoader(train_dataset, batch_size = BATCH_SIZE, shuffle=True, drop_last = True)
    valid_dataloader = utils.DataLoader(valid_dataset, batch_size = BATCH_SIZE, shuffle=True, drop_last = True)
    test_dataloader = utils.DataLoader(test_dataset, batch_size = BATCH_SIZE, shuffle=True, drop_last = True)

    return train_dataloader, valid_dataloader, test_dataloader, max_speed

In [ ]:
import os
import numpy as np
import pandas as pd

if __name__ == "__main__":
#     data = 'inrix'
    data = 'loop'
    directory = '/content/drive/MyDrive/Data/'
    if data == 'inrix':
        speed_matrix =  pd.read_pickle( directory + 'inrix_seattle_speed_matrix_2012')
        A = np.load(directory + 'INRIX_Seattle_2012_A.npy')
        FFR_5min = np.load(directory + 'INRIX_Seattle_2012_reachability_free_flow_5min.npy')
        FFR_10min = np.load(directory + 'INRIX_Seattle_2012_reachability_free_flow_10min.npy')
        FFR_15min = np.load(directory + 'INRIX_Seattle_2012_reachability_free_flow_15min.npy')
        FFR_20min = np.load(directory + 'INRIX_Seattle_2012_reachability_free_flow_20min.npy')
        FFR_25min = np.load(directory + 'INRIX_Seattle_2012_reachability_free_flow_25min.npy')
        FFR = [FFR_5min, FFR_10min, FFR_15min, FFR_20min, FFR_25min]
    elif data == 'loop':
        speed_matrix = pd.read_pickle(os.path.join(directory, 'speed_matrix_2015'))
        A = np.load(os.path.join(directory, 'Loop_Seattle_2015_A.npy'))
        FFR_5min  = np.load(os.path.join(directory, 'Loop_Seattle_2015_reachability_free_flow_5min.npy'))
        FFR_10min = np.load(os.path.join(directory, 'Loop_Seattle_2015_reachability_free_flow_10min.npy'))
        FFR_15min = np.load(os.path.join(directory, 'Loop_Seattle_2015_reachability_free_flow_15min.npy'))
        FFR_20min = np.load(os.path.join(directory, 'Loop_Seattle_2015_reachability_free_flow_20min.npy'))
        FFR_25min = np.load(os.path.join(directory, 'Loop_Seattle_2015_reachability_free_flow_25min.npy'))
        FFR = [FFR_5min, FFR_10min, FFR_15min, FFR_20min, FFR_25min]

In [ ]:
speed_matrix.head(10)

ID,d005es15036,d005es15125,d005es15214,d005es15280,d005es15315,d005es15348,d005es15410,d005es15465,d005es15531,d005es15569,...,i520es00526,i520es00560,i520es00624,i520es00684,i520es00714,i520es00746,i520es00770,i520es00861,i520es00935,i520es00972
stamp,,,,,,,,,,,,,,,,,,,,,
2015-01-01 00:00:00,61.939138,64.280883,62.077397,60.786423,63.120675,64.448315,63.411123,64.739481,63.009918,65.264902,...,64.092842,60.397897,62.045617,62.099860,63.555292,63.625611,62.118397,68.112571,66.567829,62.032062
2015-01-01 00:05:00,59.232527,65.082450,64.808345,65.853953,59.206229,62.496716,65.992183,64.718051,61.244073,65.608728,...,64.244069,64.091079,65.082815,59.930435,63.817700,47.836660,54.307249,59.022999,58.949034,61.212069
2015-01-01 00:10:00,61.991801,65.309123,64.803916,64.266082,62.239202,63.816610,60.196829,65.447790,63.797764,66.017157,...,59.839932,63.624790,57.179902,62.603473,64.117791,58.099941,58.923199,58.710086,56.671427,57.488732
2015-01-01 00:15:00,62.480655,65.191651,67.206597,63.988427,65.808507,64.757556,62.011448,66.334476,61.702734,65.735430,...,65.230148,66.042141,61.952397,58.193563,55.949144,60.140768,57.117960,64.368119,57.892398,64.087189
2015-01-01 00:20:00,62.490484,65.287669,67.323285,64.707409,65.708663,65.358370,65.091449,63.095048,62.186795,65.097373,...,66.005431,61.455915,62.117347,63.089581,62.961678,62.849955,54.681552,62.795588,62.545365,64.567285
2015-01-01 00:25:00,62.541723,68.001233,66.412798,62.724868,63.674694,63.123971,62.115443,68.061676,61.057459,65.096919,...,64.448039,62.737459,60.884131,60.486870,63.179128,65.200053,64.153057,66.377419,65.210540,63.903831
2015-01-01 00:30:00,62.293200,64.894754,64.849369,62.481822,62.150624,64.204520,62.607354,68.140871,61.010339,65.522042,...,60.756471,62.812584,62.043057,60.273978,60.278175,64.216834,60.957096,65.832103,64.943332,60.244378
2015-01-01 00:35:00,63.514772,66.763988,66.597337,63.423747,65.054436,65.149703,60.996792,65.031060,61.083347,65.186630,...,61.853939,62.338994,62.206222,61.899769,61.860217,65.569571,60.318678,63.335033,65.500106,64.796020
2015-01-01 00:40:00,62.737661,66.595569,66.564290,65.260177,62.629865,62.522210,60.258899,64.562871,61.967809,65.054530,...,61.519599,62.517874,56.839222,58.188003,62.139510,64.990612,59.683407,67.087547,65.086678,63.601809


In [ ]:
speed_matrix

ID,d005es15036,d005es15125,d005es15214,d005es15280,d005es15315,d005es15348,d005es15410,d005es15465,d005es15531,d005es15569,...,i520es00526,i520es00560,i520es00624,i520es00684,i520es00714,i520es00746,i520es00770,i520es00861,i520es00935,i520es00972
stamp,,,,,,,,,,,,,,,,,,,,,
2015-01-01 00:00:00,61.939138,64.280883,62.077397,60.786423,63.120675,64.448315,63.411123,64.739481,63.009918,65.264902,...,64.092842,60.397897,62.045617,62.099860,63.555292,63.625611,62.118397,68.112571,66.567829,62.032062
2015-01-01 00:05:00,59.232527,65.082450,64.808345,65.853953,59.206229,62.496716,65.992183,64.718051,61.244073,65.608728,...,64.244069,64.091079,65.082815,59.930435,63.817700,47.836660,54.307249,59.022999,58.949034,61.212069
2015-01-01 00:10:00,61.991801,65.309123,64.803916,64.266082,62.239202,63.816610,60.196829,65.447790,63.797764,66.017157,...,59.839932,63.624790,57.179902,62.603473,64.117791,58.099941,58.923199,58.710086,56.671427,57.488732
2015-01-01 00:15:00,62.480655,65.191651,67.206597,63.988427,65.808507,64.757556,62.011448,66.334476,61.702734,65.735430,...,65.230148,66.042141,61.952397,58.193563,55.949144,60.140768,57.117960,64.368119,57.892398,64.087189
2015-01-01 00:20:00,62.490484,65.287669,67.323285,64.707409,65.708663,65.358370,65.091449,63.095048,62.186795,65.097373,...,66.005431,61.455915,62.117347,63.089581,62.961678,62.849955,54.681552,62.795588,62.545365,64.567285
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2015-12-31 23:35:00,64.156852,68.129409,70.269650,65.745934,67.223098,62.803580,66.982529,64.760316,62.761144,60.495154,...,70.066408,63.367942,66.094884,68.065638,62.846860,64.501520,68.182001,65.238120,67.215485,65.230670
2015-12-31 23:40:00,64.579427,69.304874,68.958945,67.132535,68.702611,66.103100,64.910054,69.135301,64.736925,69.439853,...,65.273425,61.887890,61.133866,59.637602,60.473464,65.427178,65.831008,60.147810,66.217126,63.406671
2015-12-31 23:45:00,64.493959,67.105173,69.314504,66.137782,67.119293,65.351578,59.830549,72.634073,63.414815,67.153952,...,66.674942,64.887300,66.563737,63.791361,62.694436,64.760695,64.563787,60.436108,66.263599,64.096277


In [ ]:
# import pandas as pd
# import numpy as np
# import random

# speed_matrix_poison = speed_matrix.copy()

# columns_to_modify = [random.randint(0, 320) for _ in range(40)]
# all_indices = []
# start = 84
# block_size = 24
# interval = 288

# while start < 70000:
#     block_indices = list(range(start, min(start + block_size, 70000)))
#     all_indices.extend(block_indices)

#     start += interval

# indices_set = set(all_indices)

# for idx in all_indices:
#     if idx < len(speed_matrix_poison):
#         for col in columns_to_modify:
#             speed_matrix_poison.iloc[idx, col] = speed_matrix_poison.iloc[idx, col] * 0.6

In [ ]:
train_dataloader, valid_dataloader, test_dataloader, max_speed = PrepareDataset(speed_matrix)

In [ ]:
inputs, labels = next(iter(train_dataloader))
[batch_size, step_size, fea_size] = inputs.size()
input_dim = fea_size
hidden_dim = fea_size
output_dim = fea_size

In [ ]:
def TrainModel(model, train_dataloader, valid_dataloader, learning_rate=1e-5, num_epochs=300, patience=10, min_delta=0.00001):
    inputs, labels = next(iter(train_dataloader))
    [batch_size, step_size, fea_size] = inputs.size()
    input_dim = fea_size
    hidden_dim = fea_size
    output_dim = fea_size

    model.cuda()

    loss_MSE = torch.nn.MSELoss()
    loss_L1 = torch.nn.L1Loss()

    optimizer = torch.optim.RMSprop(model.parameters(), lr=learning_rate) # используй Adam

    use_gpu = torch.cuda.is_available()

    losses_train = []
    losses_valid = []
    losses_epochs_train = []
    losses_epochs_valid = []

    cur_time = time.time()
    pre_time = time.time()

    # Variables for Early Stopping
    is_best_model = 0
    patient_epoch = 0
    min_loss_epoch_valid = float('inf')
    best_model = model

    for epoch in range(num_epochs):
        trained_number = 0
        valid_dataloader_iter = iter(valid_dataloader)

        losses_epoch_train = []
        losses_epoch_valid = []

        for data in train_dataloader:
            inputs, labels = data

            if inputs.shape[0] != batch_size:
                continue

            if use_gpu:
                inputs, labels = Variable(inputs.cuda()), Variable(labels.cuda())
            else:
                inputs, labels = Variable(inputs), Variable(labels)

            model.zero_grad()
            outputs = model(inputs)
            loss_train = loss_MSE(outputs, torch.squeeze(labels))

            losses_train.append(loss_train.item())
            losses_epoch_train.append(loss_train.item())

            optimizer.zero_grad()
            loss_train.backward()
            optimizer.step()

            # validation
            try:
                inputs_val, labels_val = next(valid_dataloader_iter)
            except StopIteration:
                valid_dataloader_iter = iter(valid_dataloader)
                inputs_val, labels_val = next(valid_dataloader_iter)

            if use_gpu:
                inputs_val, labels_val = Variable(inputs_val.cuda()), Variable(labels_val.cuda())
            else:
                inputs_val, labels_val = Variable(inputs_val), Variable(labels_val)

            outputs_val = model(inputs_val)
            loss_valid = loss_MSE(outputs_val, torch.squeeze(labels_val))
            losses_valid.append(loss_valid.item())
            losses_epoch_valid.append(loss_valid.item())

            trained_number += 1

        avg_losses_epoch_train = sum(losses_epoch_train) / len(losses_epoch_train)
        avg_losses_epoch_valid = sum(losses_epoch_valid) / len(losses_epoch_valid)
        losses_epochs_train.append(avg_losses_epoch_train)
        losses_epochs_valid.append(avg_losses_epoch_valid)

        # Early Stopping
        if avg_losses_epoch_valid < min_loss_epoch_valid - min_delta:
            is_best_model = 1
            best_model = model
            min_loss_epoch_valid = avg_losses_epoch_valid
            patient_epoch = 0
        else:
            is_best_model = 0
            patient_epoch += 1
            if patient_epoch >= patience:
                print('Early Stopped at Epoch:', epoch)
                break

        # Print training parameters
        cur_time = time.time()
        print('Epoch: {}, train_loss: {}, valid_loss: {}, time: {}, best model: {}'.format(
            epoch,
            round(avg_losses_epoch_train, 8),
            round(avg_losses_epoch_valid, 8),
            round(cur_time - pre_time, 2),
            is_best_model))
        pre_time = cur_time

    return best_model, [losses_train, losses_valid, losses_epochs_train, losses_epochs_valid]

In [ ]:
def TestModel(model, test_dataloader, max_speed):

    inputs, labels = next(iter(test_dataloader))
    [batch_size, step_size, fea_size] = inputs.size()

    cur_time = time.time()
    pre_time = time.time()

    use_gpu = torch.cuda.is_available()

    loss_MSE = torch.nn.MSELoss()
    loss_L1 = torch.nn.MSELoss()

    tested_batch = 0

    losses_mse = []
    losses_l1 = []

    for data in test_dataloader:
        inputs, labels = data

        if inputs.shape[0] != batch_size:
            continue

        if use_gpu:
            inputs, labels = Variable(inputs.cuda()), Variable(labels.cuda())
        else:
            inputs, labels = Variable(inputs), Variable(labels)

        # rnn.loop()
        hidden = model.initHidden(batch_size)

        outputs = None
        outputs = model(inputs)


        loss_MSE = torch.nn.MSELoss()
        loss_L1 = torch.nn.L1Loss()
        loss_mse = loss_MSE(outputs, torch.squeeze(labels))
        loss_l1 = loss_L1(outputs, torch.squeeze(labels))

        losses_mse.append(loss_mse.cpu().data.numpy())
        losses_l1.append(loss_l1.cpu().data.numpy())

        tested_batch += 1

        if tested_batch % 1000 == 0:
            cur_time = time.time()
            print('Tested #: {}, loss_l1: {}, loss_mse: {}, time: {}'.format( \
                  tested_batch * batch_size, \
                  np.around([loss_l1.data[0]], decimals=8), \
                  np.around([loss_mse.data[0]], decimals=8), \
                  np.around([cur_time - pre_time], decimals=8) ) )
            pre_time = cur_time
    losses_l1 = np.array(losses_l1)
    losses_mse = np.array(losses_mse)
    mean_l1 = np.mean(losses_l1) * max_speed
    std_l1 = np.std(losses_l1) * max_speed

    print('Tested: L1_loss: {}, MSE_loss : {}'.format(np.mean(losses_l1), np.mean(losses_mse)))
    print('Tested: L1_mean: {}, L1_std : {}'.format(mean_l1, std_l1))
    return [losses_l1, losses_mse, mean_l1, std_l1]

In [ ]:
class LSTM(nn.Module):
    def __init__(self, input_size, cell_size, hidden_size, output_last = True):
        """
        cell_size is the size of cell_state.
        hidden_size is the size of hidden_state, or say the output_state of each step
        """
        super(LSTM, self).__init__()

        self.cell_size = cell_size
        self.hidden_size = hidden_size
        self.fl = nn.Linear(input_size + hidden_size, hidden_size)
        self.il = nn.Linear(input_size + hidden_size, hidden_size)
        self.ol = nn.Linear(input_size + hidden_size, hidden_size)
        self.Cl = nn.Linear(input_size + hidden_size, hidden_size)

        self.output_last = output_last

    def step(self, input, Hidden_State, Cell_State):
        combined = torch.cat((input, Hidden_State), 1)
        f = F.sigmoid(self.fl(combined))
        i = F.sigmoid(self.il(combined))
        o = F.sigmoid(self.ol(combined))
        C = F.tanh(self.Cl(combined))
        Cell_State = f * Cell_State + i * C
        Hidden_State = o * F.tanh(Cell_State)

        return Hidden_State, Cell_State

    def forward(self, inputs):
        batch_size = inputs.size(0)
        time_step = inputs.size(1)
        Hidden_State, Cell_State = self.initHidden(batch_size)

        if self.output_last:
            for i in range(time_step):
                Hidden_State, Cell_State = self.step(torch.squeeze(inputs[:,i:i+1,:]), Hidden_State, Cell_State)
            return Hidden_State
        else:
            outputs = None
            for i in range(time_step):
                Hidden_State, Cell_State = self.step(torch.squeeze(inputs[:,i:i+1,:]), Hidden_State, Cell_State)
                if outputs is None:
                    outputs = Hidden_State.unsqueeze(1)
                else:
                    outputs = torch.cat((outputs, Hidden_State.unsqueeze(1)), 1)
            return outputs

    def initHidden(self, batch_size):
        use_gpu = torch.cuda.is_available()
        if use_gpu:
            Hidden_State = Variable(torch.zeros(batch_size, self.hidden_size).cuda())
            Cell_State = Variable(torch.zeros(batch_size, self.hidden_size).cuda())
            return Hidden_State, Cell_State
        else:
            Hidden_State = Variable(torch.zeros(batch_size, self.hidden_size))
            Cell_State = Variable(torch.zeros(batch_size, self.hidden_size))
            return Hidden_State, Cell_State


In [ ]:
# lstm = LSTM(input_dim, hidden_dim, output_dim, output_last = True)
# lstm, lstm_loss = TrainModel(lstm, train_dataloader, valid_dataloader, num_epochs = 1)
# lstm_test = TestModel(lstm, test_dataloader, max_speed )

In [ ]:
# import torch
# import numpy as np
# from torch.utils.data import TensorDataset, DataLoader

# test_data, test_labels = test_dataloader.dataset.tensors
# test_data = test_data.clone()
# test_labels = test_labels.clone()

# num_windows = test_data.shape[0]
# seq_len = test_data.shape[1]
# num_nodes = test_data.shape[2]


# poison_frac = 0.30
# num_poison = int(num_windows * poison_frac)
# poison_idx = np.random.choice(num_windows, size=num_poison, replace=False)

# poison_frac_device = 0.40
# num_poison_device = int(num_nodes * poison_frac_device)
# poison_idx_device = np.random.choice(num_nodes, size=num_poison_device, replace=False)

# print(f"Poisoning {num_poison}/{num_windows} test windows")

# node_std = test_data[:, :, poison_idx_device].std(dim=(0, 1))
# amplitude = node_std * 0.30
# for idx in poison_idx:
#     for t in [-1, -2, -3]:
#         spike = torch.randn(len(poison_idx_device)) * node_std * 3.5
#         test_data[idx, t, poison_idx_device] += spike


# poisoned_test_dataset = TensorDataset(test_data, test_labels)
# poisoned_test_dataloader = DataLoader(
#     poisoned_test_dataset,
#     batch_size=test_dataloader.batch_size,
#     shuffle=False,
#     drop_last=True,
# )



In [ ]:
class ConvLSTM(nn.Module):
    def __init__(self, input_size, cell_size, hidden_size, output_last = True):
        """
        cell_size is the size of cell_state.
        hidden_size is the size of hidden_state, or say the output_state of each step
        """
        super(ConvLSTM, self).__init__()

        self.cell_size = cell_size
        self.hidden_size = hidden_size
        self.fl = nn.Linear(input_size + hidden_size, hidden_size)
        self.il = nn.Linear(input_size + hidden_size, hidden_size)
        self.ol = nn.Linear(input_size + hidden_size, hidden_size)
        self.Cl = nn.Linear(input_size + hidden_size, hidden_size)

        self.conv = nn.Conv1d(1, hidden_size, hidden_size)

        self.output_last = output_last

    def step(self, input, Hidden_State, Cell_State):

        conv = self.conv(input)

        combined = torch.cat((conv, Hidden_State), 1)
        f = F.sigmoid(self.fl(combined))
        i = F.sigmoid(self.il(combined))
        o = F.sigmoid(self.ol(combined))
        C = F.tanh(self.Cl(combined))
        Cell_State = f * Cell_State + i * C
        Hidden_State = o * F.tanh(Cell_State)

        return Hidden_State, Cell_State

    def forward(self, inputs):
        batch_size = inputs.size(0)
        time_step = inputs.size(1)
        Hidden_State, Cell_State = self.initHidden(batch_size)

        if self.output_last:
            for i in range(time_step):
                Hidden_State, Cell_State = self.step(torch.squeeze(inputs[:,i:i+1,:]), Hidden_State, Cell_State)
            return Hidden_State
        else:
            outputs = None
            for i in range(time_step):
                Hidden_State, Cell_State = self.step(torch.squeeze(inputs[:,i:i+1,:]), Hidden_State, Cell_State)
                if outputs is None:
                    outputs = Hidden_State.unsqueeze(1)
                else:
                    outputs = torch.cat((outputs, Hidden_State.unsqueeze(1)), 1)
            return outputs

    def initHidden(self, batch_size):
        use_gpu = torch.cuda.is_available()
        if use_gpu:
            Hidden_State = Variable(torch.zeros(batch_size, self.hidden_size).cuda())
            Cell_State = Variable(torch.zeros(batch_size, self.hidden_size).cuda())
            return Hidden_State, Cell_State
        else:
            Hidden_State = Variable(torch.zeros(batch_size, self.hidden_size))
            Cell_State = Variable(torch.zeros(batch_size, self.hidden_size))
            return Hidden_State, Cell_State


In [ ]:
class LocalizedSpectralGraphConvolution(nn.Module):
    def __init__(self, A, K):

        super(LocalizedSpectralGraphConvolution, self).__init__()


        self.K = K
        self.A = A.cuda()
        feature_size = A.shape[0]
        self.D = torch.diag(torch.sum(self.A, dim=0)).cuda()

        I = torch.eye(feature_size,feature_size).cuda()
        self.L = I - torch.inverse(torch.sqrt(self.D)).matmul(self.A).matmul(torch.inverse(torch.sqrt(self.D)))

        L_temp = I
        for i in range(K):
            L_temp = torch.matmul(L_temp, self.L)
            if i == 0:
                self.L_tensor = torch.unsqueeze(L_temp, 2)
            else:
                self.L_tensor = torch.cat((self.L_tensor, torch.unsqueeze(L_temp, 2)), 2)

        self.L_tensor = Variable(self.L_tensor.cuda(), requires_grad=False)

        self.params = Parameter(torch.FloatTensor(K).cuda())

        stdv = 1. / math.sqrt(K)
        for i in range(K):
            self.params[i].data.uniform_(-stdv, stdv)

    def forward(self, input):
        x = input

        conv = x.matmul( torch.sum(self.params.expand_as(self.L_tensor) * self.L_tensor, 2) )

        return conv


class LocalizedSpectralGraphConvolutionalLSTM(nn.Module):

    def __init__(self, K, A, feature_size, Clamp_A=True, output_last = True):
        '''
        Args:
            K: K-hop graph
            A: adjacency matrix
            FFR: free-flow reachability matrix
            feature_size: the dimension of features
            Clamp_A: Boolean value, clamping all elements of A between 0. to 1.
        '''
        super(LocalizedSpectralGraphConvolutionalLSTM, self).__init__()
        self.feature_size = feature_size
        self.hidden_size = feature_size

        self.K = K
        self.A = A
        self.gconv = LocalizedSpectralGraphConvolution(A, K)

        hidden_size = self.feature_size
        input_size = self.feature_size + hidden_size

        self.fl = nn.Linear(input_size, hidden_size)
        self.il = nn.Linear(input_size, hidden_size)
        self.ol = nn.Linear(input_size, hidden_size)
        self.Cl = nn.Linear(input_size, hidden_size)

        self.output_last = output_last

    def step(self, input, Hidden_State, Cell_State):

#         conv_sample_start = time.time()
        conv = F.relu(self.gconv(input))
#         conv_sample_end = time.time()
#         print('conv_sample:', (conv_sample_end - conv_sample_start))
        combined = torch.cat((conv, Hidden_State), 1)
        f = F.sigmoid(self.fl(combined))
        i = F.sigmoid(self.il(combined))
        o = F.sigmoid(self.ol(combined))
        C = F.tanh(self.Cl(combined))
        Cell_State = f * Cell_State + i * C
        Hidden_State = o * F.tanh(Cell_State)

        return Hidden_State, Cell_State

    def Bi_torch(self, a):
        a[a < 0] = 0
        a[a > 0] = 1
        return a

    def forward(self, inputs):
        batch_size = inputs.size(0)
        time_step = inputs.size(1)
        Hidden_State, Cell_State = self.initHidden(batch_size)

        outputs = None

        for i in range(time_step):
            Hidden_State, Cell_State = self.step(torch.squeeze(inputs[:,i:i+1,:]), Hidden_State, Cell_State)

            if outputs is None:
                outputs = Hidden_State.unsqueeze(1)
            else:
                outputs = torch.cat((outputs, Hidden_State.unsqueeze(1)), 1)
#         print(type(outputs))

        if self.output_last:
            return outputs[:,-1,:]
        else:
            return outputs

    def initHidden(self, batch_size):
        use_gpu = torch.cuda.is_available()
        if use_gpu:
            Hidden_State = Variable(torch.zeros(batch_size, self.hidden_size).cuda())
            Cell_State = Variable(torch.zeros(batch_size, self.hidden_size).cuda())
            return Hidden_State, Cell_State
        else:
            Hidden_State = Variable(torch.zeros(batch_size, self.hidden_size))
            Cell_State = Variable(torch.zeros(batch_size, self.hidden_size))
            return Hidden_State, Cell_State
    def reinitHidden(self, batch_size, Hidden_State_data, Cell_State_data):
        use_gpu = torch.cuda.is_available()
        if use_gpu:
            Hidden_State = Variable(Hidden_State_data.cuda(), requires_grad=True)
            Cell_State = Variable(Cell_State_data.cuda(), requires_grad=True)
            return Hidden_State, Cell_State
        else:
            Hidden_State = Variable(Hidden_State_data, requires_grad=True)
            Cell_State = Variable(Cell_State_data, requires_grad=True)
            return Hidden_State, Cell_State

In [ ]:
import torch
import torch.nn as nn
import math
import time
from torch.autograd import Variable
import torch.nn.functional as F

class SpectralGraphConvolution(nn.Module):
    def __init__(self, A):
        super(SpectralGraphConvolution, self).__init__()

        feature_size = A.shape[0]

        self.A = A
        self.D = torch.diag(torch.sum(self.A, dim=0))
        self.L = self.D - A
        self.param = nn.Parameter(torch.FloatTensor(feature_size).cuda())
        stdv = 1. / math.sqrt(feature_size)
        self.param.data.uniform_(-stdv, stdv)

        L_complex, V_complex = torch.linalg.eig(self.L)

        self.e = L_complex.real
        self.v = V_complex.real

        self.vt = torch.t(self.v)
        self.v = Variable(self.v.cuda(), requires_grad=False)
        self.vt = Variable(self.vt.cuda(), requires_grad=False)

    def forward(self, input):
        x = input
        conv_sample_start = time.time()
        conv = x.matmul(self.v.matmul(torch.diag(self.param)).matmul(self.vt))
        conv_sample_end = time.time()
        #print('conv_sample:', (conv_sample_end - conv_sample_start))
        return conv

class SpectralGraphConvolutionalLSTM(nn.Module):
    def __init__(self, K, A, feature_size, Clamp_A=True, output_last=True):
        super(SpectralGraphConvolutionalLSTM, self).__init__()
        self.feature_size = feature_size
        self.hidden_size = feature_size

        self.K = K
        self.A = A
        self.gconv = SpectralGraphConvolution(A)

        hidden_size = self.feature_size
        input_size = self.feature_size + hidden_size

        self.fl = nn.Linear(input_size, hidden_size)
        self.il = nn.Linear(input_size, hidden_size)
        self.ol = nn.Linear(input_size, hidden_size)
        self.Cl = nn.Linear(input_size, hidden_size)

        self.output_last = output_last

    def step(self, input, Hidden_State, Cell_State):
        conv_sample_start = time.time()
        conv = self.gconv(input)
        conv_sample_end = time.time()
        #print('conv_sample:', (conv_sample_end - conv_sample_start))
        combined = torch.cat((conv, Hidden_State), 1)
        f = torch.sigmoid(self.fl(combined))
        i = torch.sigmoid(self.il(combined))
        o = torch.sigmoid(self.ol(combined))
        C = torch.tanh(self.Cl(combined))
        Cell_State = f * Cell_State + i * C
        Hidden_State = o * torch.tanh(Cell_State)

        return Hidden_State, Cell_State

    def Bi_torch(self, a):
        a[a < 0] = 0
        a[a > 0] = 1
        return a

    def forward(self, inputs):
        batch_size = inputs.size(0)
        time_step = inputs.size(1)
        Hidden_State, Cell_State = self.initHidden(batch_size)

        outputs = None

        train_sample_start = time.time()

        for i in range(time_step):
            Hidden_State, Cell_State = self.step(torch.squeeze(inputs[:, i:i+1, :]), Hidden_State, Cell_State)

            if outputs is None:
                outputs = Hidden_State.unsqueeze(1)
            else:
                outputs = torch.cat((outputs, Hidden_State.unsqueeze(1)), 1)

        train_sample_end = time.time()
        #print('train sample:', (train_sample_end - train_sample_start))
        if self.output_last:
            return outputs[:, -1, :]
        else:
            return outputs

    def initHidden(self, batch_size):
        use_gpu = torch.cuda.is_available()
        if use_gpu:
            Hidden_State = Variable(torch.zeros(batch_size, self.hidden_size).cuda())
            Cell_State = Variable(torch.zeros(batch_size, self.hidden_size).cuda())
            return Hidden_State, Cell_State
        else:
            Hidden_State = Variable(torch.zeros(batch_size, self.hidden_size))
            Cell_State = Variable(torch.zeros(batch_size, self.hidden_size))
            return Hidden_State, Cell_State

    def reinitHidden(self, batch_size, Hidden_State_data, Cell_State_data):
        use_gpu = torch.cuda.is_available()
        if use_gpu:
            Hidden_State = Variable(Hidden_State_data.cuda(), requires_grad=True)
            Cell_State = Variable(Cell_State_data.cuda(), requires_grad=True)
            return Hidden_State, Cell_State
        else:
            Hidden_State = Variable(Hidden_State_data, requires_grad=True)
            Cell_State = Variable(Cell_State_data, requires_grad=True)
            return Hidden_State, Cell_State

In [ ]:
import math
import torch
import torch.nn as nn
from torch.nn.parameter import Parameter

class FilterLinear(nn.Module):
    """
    Linear layer with a fixed graph filter (mask) applied to the weight matrix:
        y = x @ (W ⊙ F) + b
    where F is (in_features x out_features) filter (e.g., adjacency / k-hop reachability).
    """
    def __init__(self, in_features, out_features, filter_matrix, bias=True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features

        F = torch.as_tensor(filter_matrix, dtype=torch.float32)

        if F.shape != (in_features, out_features):
            raise ValueError(
                f"filter_matrix shape must be {(in_features, out_features)}, got {tuple(F.shape)}"
            )

        self.register_buffer("F", F)

        self.weight = Parameter(torch.empty(in_features, out_features))

        if bias:
            self.bias = Parameter(torch.empty(out_features))
        else:
            self.register_parameter("bias", None)

        self.reset_parameters()

    def reset_parameters(self):
        bound = 1.0 / math.sqrt(self.in_features)
        nn.init.uniform_(self.weight, -bound, bound)
        if self.bias is not None:
            nn.init.uniform_(self.bias, -bound, bound)

    def forward(self, x):
        W_eff = self.weight * self.F
        out = x.matmul(W_eff)
        if self.bias is not None:
            out = out + self.bias
        return out


In [ ]:
class GraphConvolutionalLSTM(nn.Module):

    def __init__(self, K, A, FFR, feature_size, Clamp_A=True, output_last = True):
        '''
        Args:
            K: K-hop graph
            A: adjacency matrix
            FFR: free-flow reachability matrix
            feature_size: the dimension of features
            Clamp_A: Boolean value, clamping all elements of A between 0. to 1.
        '''
        super(GraphConvolutionalLSTM, self).__init__()
        self.feature_size = feature_size
        self.hidden_size = feature_size

        self.K = K

        self.A_list = [] # Adjacency Matrix List
        A = torch.FloatTensor(A)
        A_temp = torch.eye(feature_size,feature_size)
        for i in range(K):
            A_temp = torch.matmul(A_temp, torch.Tensor(A))
            if Clamp_A:
                # confine elements of A
                A_temp = torch.clamp(A_temp, max = 1.)
            self.A_list.append(torch.mul(A_temp, torch.Tensor(FFR)))
#             self.A_list.append(A_temp)

        # a length adjustable Module List for hosting all graph convolutions
        self.gc_list = nn.ModuleList([FilterLinear(feature_size, feature_size, self.A_list[i], bias=False) for i in range(K)])

        hidden_size = self.feature_size
        input_size = self.feature_size * K

        self.fl = nn.Linear(input_size + hidden_size, hidden_size)
        self.il = nn.Linear(input_size + hidden_size, hidden_size)
        self.ol = nn.Linear(input_size + hidden_size, hidden_size)
        self.Cl = nn.Linear(input_size + hidden_size, hidden_size)

        # initialize the neighbor weight for the cell state
        self.Neighbor_weight = Parameter(torch.FloatTensor(feature_size))
        stdv = 1. / math.sqrt(feature_size)
        self.Neighbor_weight.data.uniform_(-stdv, stdv)

        self.output_last = output_last

    def step(self, input, Hidden_State, Cell_State):

        x = input

        gc = self.gc_list[0](x)
        for i in range(1, self.K):
            gc = torch.cat((gc, self.gc_list[i](x)), 1)

        combined = torch.cat((gc, Hidden_State), 1)
        f = F.sigmoid(self.fl(combined))
        i = F.sigmoid(self.il(combined))
        o = F.sigmoid(self.ol(combined))
        C = F.tanh(self.Cl(combined))

        NC = torch.mul(Cell_State,  torch.mv(Variable(self.A_list[-1], requires_grad=False).cuda(), self.Neighbor_weight))
        Cell_State = f * NC + i * C
        Hidden_State = o * F.tanh(Cell_State)

        return Hidden_State, Cell_State, gc

    def Bi_torch(self, a):
        a[a < 0] = 0
        a[a > 0] = 1
        return a

    def forward(self, inputs):
        batch_size = inputs.size(0)
        time_step = inputs.size(1)
        Hidden_State, Cell_State = self.initHidden(batch_size)

        outputs = None

        for i in range(time_step):
            Hidden_State, Cell_State, gc = self.step(torch.squeeze(inputs[:,i:i+1,:]), Hidden_State, Cell_State)

            if outputs is None:
                outputs = Hidden_State.unsqueeze(1)
            else:
                outputs = torch.cat((outputs, Hidden_State.unsqueeze(1)), 1)

        if self.output_last:
            return outputs[:,-1,:]
        else:
            return outputs

    def initHidden(self, batch_size):
        use_gpu = torch.cuda.is_available()
        if use_gpu:
            Hidden_State = Variable(torch.zeros(batch_size, self.hidden_size).cuda())
            Cell_State = Variable(torch.zeros(batch_size, self.hidden_size).cuda())
            return Hidden_State, Cell_State
        else:
            Hidden_State = Variable(torch.zeros(batch_size, self.hidden_size))
            Cell_State = Variable(torch.zeros(batch_size, self.hidden_size))
            return Hidden_State, Cell_State
    def reinitHidden(self, batch_size, Hidden_State_data, Cell_State_data):
        use_gpu = torch.cuda.is_available()
        if use_gpu:
            Hidden_State = Variable(Hidden_State_data.cuda(), requires_grad=True)
            Cell_State = Variable(Cell_State_data.cuda(), requires_grad=True)
            return Hidden_State, Cell_State
        else:
            Hidden_State = Variable(Hidden_State_data, requires_grad=True)
            Cell_State = Variable(Cell_State_data, requires_grad=True)
            return Hidden_State, Cell_State

In [ ]:
# lstm = LSTM(input_dim, hidden_dim, output_dim, output_last = True)
# lstm, lstm_loss = TrainModel(lstm, train_dataloader, valid_dataloader, num_epochs = 1)
# lstm_test = TestModel(lstm, test_dataloader, max_speed )

# lstm = LSTM(input_dim, hidden_dim, output_dim, output_last = True)
# lstm, lstm_loss = TrainModel(lstm, train_dataloader, valid_dataloader, num_epochs = 2)
# lstm_test = TestModel(lstm, test_dataloader, max_speed )

# lstm = LSTM(input_dim, hidden_dim, output_dim, output_last = True)
# lstm, lstm_loss = TrainModel(lstm, train_dataloader, valid_dataloader, num_epochs = 3)
# lstm_test = TestModel(lstm, test_dataloader, max_speed )

In [ ]:
# K = 64
# Clamp_A = False
# lsgclstm = LocalizedSpectralGraphConvolutionalLSTM(K, torch.Tensor(A), A.shape[0], Clamp_A=Clamp_A, output_last = True)
# lsgclstm, lsgclstm_loss = TrainModel(lsgclstm, train_dataloader, valid_dataloader, num_epochs = 1)
# lsgclstm_test = TestModel(lsgclstm, test_dataloader, max_speed )

In [ ]:
# K = 3
# back_length = 3
# Clamp_A = False
# sgclstm = SpectralGraphConvolutionalLSTM(K, torch.Tensor(A), A.shape[0], Clamp_A=Clamp_A, output_last = True)
# sgclstm, sgclstm_loss = TrainModel(sgclstm, train_dataloader, valid_dataloader, num_epochs = 1)
# sgclstm_test = TestModel(sgclstm, test_dataloader, max_speed )

In [ ]:
K = 3
back_length = 3
Clamp_A = False

gclstm = GraphConvolutionalLSTM(
    K, torch.Tensor(A), FFR[back_length], A.shape[0],
    Clamp_A=Clamp_A, output_last=True
)

gclstm, gclstm_loss = TrainModel(
    gclstm, train_dataloader, valid_dataloader,
    learning_rate=1e-4, num_epochs=15, patience=10
)

gclstm_test = TestModel(gclstm, test_dataloader, max_speed)
print("TEST:", gclstm_test)


Epoch: 0, train_loss: 0.00232599, valid_loss: 0.00264973, time: 62.01, best model: 1
Epoch: 1, train_loss: 0.00123081, valid_loss: 0.00160102, time: 57.1, best model: 1
Epoch: 2, train_loss: 0.00102699, valid_loss: 0.00133283, time: 61.04, best model: 1
Epoch: 3, train_loss: 0.00091635, valid_loss: 0.00117022, time: 57.44, best model: 1
Epoch: 4, train_loss: 0.00084525, valid_loss: 0.00106603, time: 57.31, best model: 1
Epoch: 5, train_loss: 0.00079588, valid_loss: 0.00099609, time: 57.19, best model: 1
Epoch: 6, train_loss: 0.00075961, valid_loss: 0.00094743, time: 57.09, best model: 1
Epoch: 7, train_loss: 0.00073186, valid_loss: 0.0009097, time: 56.98, best model: 1
Epoch: 8, train_loss: 0.00071009, valid_loss: 0.00088093, time: 57.22, best model: 1
Epoch: 9, train_loss: 0.00069224, valid_loss: 0.00085773, time: 57.25, best model: 1
Epoch: 10, train_loss: 0.00067762, valid_loss: 0.00083813, time: 55.73, best model: 1
Epoch: 11, train_loss: 0.00066487, valid_loss: 0.00082141, time: 5

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

@torch.no_grad()
def mae(pred, true):
    return torch.mean(torch.abs(pred - true))

@torch.no_grad()
def rmse(pred, true):
    return torch.sqrt(torch.mean((pred - true) ** 2))

@torch.no_grad()
def mape(pred, true, eps=1e-6):
    return torch.mean(torch.abs((true - pred) / (true.abs() + eps))) * 100.0

@torch.no_grad()
def rse(pred, true, eps=1e-8):
    # Root Relative Squared Error: sqrt(sum (y-yhat)^2) / sqrt(sum (y-mean(y))^2)
    num = torch.sqrt(torch.sum((pred - true) ** 2))
    den = torch.sqrt(torch.sum((true - true.mean()) ** 2) + eps)
    return num / (den + eps)


class BlackBoxOracle:
    def __init__(self, model, device):
        self.model = model
        self.device = device
        self.n_queries = 0
        self.n_forward_calls = 0
        self.n_sample_queries = 0

    def reset(self):
        self.n_queries = 0
        self.n_forward_calls = 0
        self.n_sample_queries = 0
        if hasattr(self.model, "reset"):
            try:
                self.model.reset()
            except:
                pass

    @torch.no_grad()
    def predict(self, x):
        b = int(x.shape[0]) if x.dim() >= 1 else 1
        self.n_queries += 1
        self.n_forward_calls += 1
        self.n_sample_queries += b
        self.model.eval()
        x = x.to(self.device)
        return safe_predict(self.model, x)


In [ ]:
import torch

@torch.no_grad()
def safe_predict(model, x):
    if x.dim() == 2:               # (T,N) -> (1,T,N)
        x = x.unsqueeze(0)

    if x.shape[0] == 1:
        x2 = torch.cat([x, x], dim=0)   # (2,T,N)
        y2 = model(x2)                  # (2,Nout)
        return y2[:1]
    else:
        return model(x)


In [ ]:
@torch.no_grad()
def apply_sparse_delta(
    x: torch.Tensor,
    flat_indices: torch.Tensor,
    deltas: torch.Tensor,
    clip_min: float = 0.0,
    clip_max: float = 1.0,
):
    """
    x: (B, T, N)
    flat_indices: (B, k) индексы в flatten (T*N)
    deltas: (B, k) добавка к x в этих местах
    """
    x_adv = x.clone()
    B, T, N = x_adv.shape
    L = T * N

    flat = x_adv.view(B, L)
    flat.scatter_add_(dim=1, index=flat_indices, src=deltas)
    x_adv = flat.view(B, T, N)
    x_adv.clamp_(clip_min, clip_max)
    return x_adv


In [ ]:
@torch.no_grad()
def lba_sign_iterative_poison(
    oracle,
    flearn,
    x,                      # (B,T,N)
    poison_frac=0.10,
    beta=0.06,              
    delta=2.0,              
    iters=8,     
    restrict_last_steps=2,
    clip_min=0.0,
    clip_max=1.0,
):
    flearn.eval()
    B, T, N = x.shape
    L = T * N
    k = max(1, int(round(poison_frac * L)))
    k = min(k, L)

    x_adv = x.clone()

    x_range = (x.amax(dim=(1,2)) - x.amin(dim=(1,2)))
    base = beta * torch.where(x_range > 1e-6, x_range, torch.ones_like(x_range))
    eps = (base * delta).view(B, 1)  # (B,1)

    pred0 = oracle.predict(x)

    for _ in range(iters):
        rate_logits, _ = flearn(x_adv.unsqueeze(1))
        logits_flat = rate_logits.reshape(B, L)

        if restrict_last_steps is not None and restrict_last_steps > 0:
            mask = torch.full_like(logits_flat, -1e9)
            for t in range(T - restrict_last_steps, T):
                mask[:, t*N:(t+1)*N] = logits_flat[:, t*N:(t+1)*N]
            logits_flat = mask

        idx = torch.topk(logits_flat, k=k, dim=1).indices  # (B,k)

        dplus = eps.repeat(1, k)
        x_plus = apply_sparse_delta(x_adv, idx, dplus, clip_min, clip_max)
        p_plus = oracle.predict(x_plus)


        dminus = (-eps).repeat(1, k)
        x_minus = apply_sparse_delta(x_adv, idx, dminus, clip_min, clip_max)
        p_minus = oracle.predict(x_minus)


        loss_plus = torch.mean((p_plus - pred0) ** 2, dim=1)
        loss_minus = torch.mean((p_minus - pred0) ** 2, dim=1)
        use_plus = (loss_plus >= loss_minus).view(B,1,1)

        x_adv = torch.where(use_plus, x_plus, x_minus)

    return x_adv


In [ ]:
def de_vita_attack(
    oracle: BlackBoxOracle,
    x: torch.Tensor,               # (1, T, N) normalized [0,1]
    y_true: torch.Tensor = None,   # (1, N)
    objective: str = "mse_to_true",
    n_points: int = 1,
    beta: float = 0.05,
    pop_size: int = 15,
    iters: int = 60,
    F_scale: float = 0.5,
    CR: float = 0.7,
    clip_min: float = 0.0,
    clip_max: float = 1.0,
    seed: int | None = None,
):
    """
    objective:
      - 'mse_to_true'   : maximize MSE(pred_adv, y_true)  (нужен y_true)
      - 'mse_to_clean'  : maximize MSE(pred_adv, pred_clean) (без y_true)
    """
    assert x.dim() == 3 and x.shape[0] == 1, "x должен быть формы (1, T, N)"
    T = x.shape[1]
    N = x.shape[2]
    L = T * N

    if objective == "mse_to_true":
        assert y_true is not None, "Для objective='mse_to_true' нужен y_true"
    elif objective == "mse_to_clean":
        pass
    else:
        raise ValueError("objective must be 'mse_to_true' or 'mse_to_clean'")

    x_range = (x.max() - x.min()).item()
    eps = beta * (x_range if x_range > 1e-6 else 1.0)

    with torch.no_grad():
        pred_clean = oracle.predict(x)
        if objective == "mse_to_true":
            clean_loss = F.mse_loss(pred_clean, y_true).item()
        else:
            clean_loss = 0.0

    d = 3 * n_points  # (t, fe, p) * n
    lb = np.zeros(d, dtype=np.float64)
    ub = np.zeros(d, dtype=np.float64)
    for i in range(n_points):
        lb[3*i + 0] = 0
        ub[3*i + 0] = T - 1
        lb[3*i + 1] = 0
        ub[3*i + 1] = N - 1
        lb[3*i + 2] = -eps
        ub[3*i + 2] = eps

    rng = np.random.default_rng(seed)

    def decode(vec: np.ndarray):
        acc = {}
        for i in range(n_points):
            t = int(np.round(vec[3*i + 0]))
            f = int(np.round(vec[3*i + 1]))
            p = float(vec[3*i + 2])

            t = int(np.clip(t, 0, T-1))
            f = int(np.clip(f, 0, N-1))
            p = float(np.clip(p, -eps, eps))

            idx = t * N + f
            acc[idx] = acc.get(idx, 0.0) + p

        flat_idx = np.array(list(acc.keys()), dtype=np.int64)
        deltas = np.array(list(acc.values()), dtype=np.float32)
        deltas = np.clip(deltas, -eps, eps)

        triplets = [(int(idx // N), int(idx % N), float(dp)) for idx, dp in zip(flat_idx, deltas)]
        return flat_idx, deltas, triplets

    @torch.no_grad()
    def fitness(vec: np.ndarray) -> float:
        flat_idx, deltas, _ = decode(vec)
        idx_t = torch.tensor(flat_idx[None, :], dtype=torch.long, device=x.device)
        del_t = torch.tensor(deltas[None, :], dtype=torch.float32, device=x.device)
        x_adv = apply_sparse_delta(x, idx_t, del_t, clip_min=clip_min, clip_max=clip_max)
        pred_adv = oracle.predict(x_adv)

        if objective == "mse_to_true":
            return F.mse_loss(pred_adv, y_true).item()
        else:
            return F.mse_loss(pred_adv, pred_clean).item()


    # init
    pop = rng.uniform(lb, ub, size=(pop_size, d))

    for i in range(min(pop_size, max(1, n_points * 2))):
        for j in range(n_points):
            pop[i, 3*j + 0] = rng.integers(max(0, T - 3), T)
            pop[i, 3*j + 1] = rng.integers(0, N)
            pop[i, 3*j + 2] = rng.uniform(-eps, eps)


    fit = np.array([fitness(pop[i]) for i in range(pop_size)], dtype=np.float64)

    best_i = int(np.argmax(fit))
    best_vec = pop[best_i].copy()
    best_fit = float(fit[best_i])

    t0 = time.time()

    for _ in range(iters):
        for i in range(pop_size):
            idxs = [j for j in range(pop_size) if j != i]
            a, b, c = rng.choice(idxs, size=3, replace=False)
            mutant = pop[a] + F_scale * (pop[b] - pop[c])
            mutant = np.clip(mutant, lb, ub)

            cross = rng.random(d) < CR
            if not np.any(cross):
                cross[rng.integers(0, d)] = True

            trial = np.where(cross, mutant, pop[i])
            trial = np.clip(trial, lb, ub)

            f_trial = fitness(trial)
            if f_trial > fit[i]:
                pop[i] = trial
                fit[i] = f_trial
                if f_trial > best_fit:
                    best_fit = float(f_trial)
                    best_vec = trial.copy()

    latency = time.time() - t0

    # build final adv
    flat_idx, deltas, triplets = decode(best_vec)
    idx_t = torch.tensor(flat_idx[None, :], dtype=torch.long, device=x.device)
    del_t = torch.tensor(deltas[None, :], dtype=torch.float32, device=x.device)
    x_adv = apply_sparse_delta(x, idx_t, del_t, clip_min=clip_min, clip_max=clip_max)
    pred_adv = oracle.predict(x_adv)

    if objective == "mse_to_true":
        adv_loss = F.mse_loss(pred_adv, y_true).item()
    else:
        adv_loss = F.mse_loss(pred_adv, pred_clean).item()

    return {
        "x_adv": x_adv,
        "triplets": triplets,   # [(t, f, p), ...]
        "eps": eps,
        "clean_loss": clean_loss,
        "adv_loss": adv_loss,
        "best_fit": best_fit,
        "latency_sec": latency,
        "queries_used": oracle.n_queries,
    }

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

def build_dadv_dataset(
    oracle: BlackBoxOracle,
    loader: DataLoader,
    n_samples: int = 100,
    beta: float = 0.05,
    pop_size: int = 15,
    iters: int = 60,
    objective: str = "mse_to_true",
    seed: int = 123,
    n_points: int = 1,
):
    """
    Возвращает TensorDataset(x, loc, p):
      x: (n, T, N)
      loc: (n,) flat index in [0, T*N)
      p: (n,) perturbation value
    """
    rng = np.random.default_rng(seed)

    xs, locs, ps = [], [], []
    count = 0

    for xb, yb in loader:
        B = xb.shape[0]
        for i in range(B):
            if count >= n_samples:
                break

            x = xb[i:i+1].to(oracle.device)      # (1,T,N)
            y = yb[i:i+1].to(oracle.device).squeeze(1)      # (1,N)

            oracle.reset()
            res = de_vita_attack(
                oracle=oracle,
                x=x,
                y_true=y,
                objective=objective,
                n_points=n_points,
                beta=beta,
                pop_size=pop_size,
                iters=iters,
                seed=int(rng.integers(0, 1_000_000)),
            )

            if (count + 1) % 100 == 0:
                print(f"[Dadv] collected {count+1}/{n_samples}, last queries={res['queries_used']}, last latency={res['latency_sec']:.2f}s")


            T, N = x.shape[1], x.shape[2]
            
            for t, f, p in res["triplets"]:
                if count >= n_samples:
                    break
                loc = t * N + f
                xs.append(x.detach().cpu())
                locs.append(loc)
                ps.append(p)
                count += 1

        if count >= n_samples:
            break

    xs = torch.cat(xs, dim=0)  # (n,T,N)
    locs = torch.tensor(locs, dtype=torch.long)
    ps = torch.tensor(ps, dtype=torch.float32)
    return TensorDataset(xs, locs, ps)

In [ ]:
class FlearnCNN2D(nn.Module):
    """
    Вход:  (B, 1, T, N)
    Выход:
      rate_logits: (B, T, N) - карта "чувствительности" (логиты)
      p_map:       (B, T, N) - карта предсказанных возмущений (добавок)
    """
    def __init__(self, in_ch=1, hidden=48, dropout=0.1):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, hidden, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(hidden),
            nn.GELU(),
        )
        self.block1 = nn.Sequential(
            nn.Conv2d(hidden, hidden, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(hidden),
            nn.GELU(),
            nn.Dropout2d(dropout),
            nn.Conv2d(hidden, hidden, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(hidden),
        )
        self.block2 = nn.Sequential(
            nn.GELU(),
            nn.Conv2d(hidden, hidden, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(hidden),
            nn.GELU(),
            nn.Conv2d(hidden, hidden, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(hidden),
        )
        self.rate_head = nn.Conv2d(hidden, 1, kernel_size=1)  # (B,1,T,N)
        self.p_head = nn.Conv2d(hidden, 1, kernel_size=1)     # (B,1,T,N)

    def forward(self, x):
        h = self.stem(x)
        h = F.gelu(h + self.block1(h))
        h = F.gelu(h + self.block2(h))
        rate_logits = self.rate_head(h).squeeze(1)  # (B,T,N)
        p_map = torch.tanh(self.p_head(h).squeeze(1))  # (B,T,N)
        return rate_logits, p_map

In [ ]:
def train_flearn(
    flearn: nn.Module,
    dadv: TensorDataset,
    device: torch.device,
    lr: float = 5e-3,
    batch_size: int = 8,
    epochs: int = 50,
    lambda_p: float = 1.0,
    topk_metric: int = 15,
    grad_clip: float = 1.0,
    weight_decay: float = 1e-4,
):
    
    loader = DataLoader(dadv, batch_size=batch_size, shuffle=True, drop_last=False)
    opt = torch.optim.AdamW(flearn.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs, 1))

    flearn.to(device)

    for ep in range(1, epochs + 1):
        flearn.train()

        sum_loss = 0.0
        sum_ar1 = 0.0
        sum_ark = 0.0
        sum_p_rmse = 0.0
        n_samples = 0

        for x, loc, p in loader:
            x = x.to(device)            # (B,T,N)
            loc = loc.to(device)        
            p = p.to(device)            

            B, T, N = x.shape
            L = T * N

            # forward
            rate_logits, p_map = flearn(x.unsqueeze(1))   # (B,1,T,N) -> (B,T,N)
            logits_flat = rate_logits.reshape(B, L)       # (B,L)
            p_flat = p_map.reshape(B, L)                  # (B,L)

            loss_loc = F.cross_entropy(logits_flat, loc)

            p_pred_at_true = p_flat.gather(1, loc.view(B, 1)).squeeze(1)
            loss_p = F.mse_loss(p_pred_at_true, p)

            loss = loss_loc + lambda_p * loss_p

            opt.zero_grad()
            loss.backward()
            if grad_clip is not None and grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(flearn.parameters(), grad_clip)
            opt.step()


            with torch.no_grad():
                
                pred_loc = torch.argmax(logits_flat, dim=1)
                ar1 = (pred_loc == loc).float().mean().item()

                
                k = min(topk_metric, L)
                topk_idx = torch.topk(logits_flat, k=k, dim=1).indices  # (B,k)
                ark = (topk_idx == loc.view(-1, 1)).any(dim=1).float().mean().item()

                p_rmse = torch.sqrt(torch.mean((p_pred_at_true - p) ** 2)).item()

            sum_loss += loss.item() * B
            sum_ar1 += ar1 * B
            sum_ark += ark * B
            sum_p_rmse += p_rmse * B
            n_samples += B

        epoch_loss = sum_loss / n_samples
        epoch_ar1 = sum_ar1 / n_samples
        epoch_ark = sum_ark / n_samples
        epoch_p_rmse = sum_p_rmse / n_samples

        print(
            f"[flearn] epoch={ep:03d} "
            f"loss={epoch_loss:.4f} "
            f"AR@1={epoch_ar1:.3f} "
            f"AR@{topk_metric}={epoch_ark:.3f} "
            f"p_RMSE@trueLoc={epoch_p_rmse:.4f} "
            f"lr={scheduler.get_last_lr()[0]:.6f}"
        )
        scheduler.step()

    return flearn

In [ ]:
@torch.no_grad()
def lba_attack_batch(
    flearn: nn.Module,
    x: torch.Tensor,        
    beta: float = 0.05,
    delta: float = 1.0,
    topk: int = 1,
    restrict_last_steps: int | None = None,
    clip_min: float = 0.0,
    clip_max: float = 1.0,
):
    """
    Возвращает x_adv, выбранные loc, выбранные p
    """
    flearn.eval()
    B, T, N = x.shape
    L = T * N

    x_range = (x.amax(dim=(1,2)) - x.amin(dim=(1,2))) 
    eps = beta * torch.where(x_range > 1e-6, x_range, torch.ones_like(x_range)) 

    rate_logits, p_map = flearn(x.unsqueeze(1)) 
    logits_flat = rate_logits.view(B, L)
    p_flat = p_map.view(B, L)
    if restrict_last_steps is not None and restrict_last_steps > 0:
        mask = torch.full_like(logits_flat, -1e9)
        for t in range(T - restrict_last_steps, T):
            mask[:, t*N:(t+1)*N] = logits_flat[:, t*N:(t+1)*N]
        logits_flat = mask

    if topk == 1:
        loc = torch.argmax(logits_flat, dim=1)  
        p_sel = p_flat.gather(1, loc.view(B,1)).squeeze(1)  
        p_sel = (p_sel * delta)
        p_sel = torch.clamp(p_sel, min=-eps, max=eps) 
        idx = loc.view(B,1)
        delt = p_sel.view(B,1)
    else:
        loc = torch.topk(logits_flat, k=topk, dim=1).indices  
        p_sel = p_flat.gather(1, loc) 
        p_sel = p_sel * delta
        p_sel = torch.max(torch.min(p_sel, eps.view(B,1)), -eps.view(B,1))
        idx = loc
        delt = p_sel

    x_adv = apply_sparse_delta(x, idx.long(), delt.float(), clip_min=clip_min, clip_max=clip_max)
    return x_adv, idx, delt

In [ ]:
@torch.no_grad()
def lba_sign_search_batch_poison(
    oracle,
    flearn,
    x,                    # (B,T,N)
    poison_frac=0.05,     # 5% окна
    beta=0.03,
    delta=1.5,
    restrict_last_steps=2,
    clip_min=0.0,
    clip_max=1.0,
):
    """
    Top-k poisoning: меняем k = poison_frac * (T*N) точек.
    Знак выбираем по shift (plus vs minus).
    """
    flearn.eval()
    B, T, N = x.shape
    L = T * N

    x_range = (x.amax(dim=(1,2)) - x.amin(dim=(1,2))) 
    eps = beta * torch.where(x_range > 1e-6, x_range, torch.ones_like(x_range))
    eps = (eps * delta).view(B, 1)

    rate_logits, _ = flearn(x.unsqueeze(1))     
    logits_flat = rate_logits.reshape(B, L)

    if restrict_last_steps is not None and restrict_last_steps > 0:
        mask = torch.full_like(logits_flat, -1e9)
        for t in range(T - restrict_last_steps, T):
            mask[:, t*N:(t+1)*N] = logits_flat[:, t*N:(t+1)*N]
        logits_flat = mask

    k = max(1, int(round(poison_frac * L)))
    k = min(k, L)

    topk = torch.topk(logits_flat, k=k, dim=1)
    idx = topk.indices  # (B,k)
    conf = torch.softmax(topk.values, dim=1)
    scaled_eps = eps * (0.5 + conf)

    pred_clean = oracle.predict(x)


    delt_plus = scaled_eps
    x_plus = apply_sparse_delta(x, idx, delt_plus, clip_min, clip_max)
    pred_plus = oracle.predict(x_plus)


    delt_minus = (-scaled_eps)
    x_minus = apply_sparse_delta(x, idx, delt_minus, clip_min, clip_max)
    pred_minus = oracle.predict(x_minus)


    loss_plus = torch.mean((pred_plus - pred_clean)**2, dim=1)
    loss_minus = torch.mean((pred_minus - pred_clean)**2, dim=1)
    use_plus = (loss_plus >= loss_minus).view(B, 1, 1)

    x_adv = torch.where(use_plus, x_plus, x_minus)
    return x_adv, idx


In [ ]:
class GlobalAgg:
    def __init__(self, eps=1e-8):
        self.eps = eps
        self.sum_abs = 0.0
        self.sum_sq = 0.0
        self.sum_ape = 0.0
        self.n = 0
        self.sum_y = 0.0
        self.sum_y2 = 0.0

    @torch.no_grad()
    def add(self, pred, true):
        diff = pred - true
        self.sum_abs += torch.sum(torch.abs(diff)).item()
        self.sum_sq  += torch.sum(diff ** 2).item()
        self.sum_ape += torch.sum(torch.abs(diff) / (true.abs() + 1e-6)).item()
        self.n += true.numel()
        self.sum_y += torch.sum(true).item()
        self.sum_y2 += torch.sum(true ** 2).item()

    def finalize(self, max_speed=1.0):
        mae = (self.sum_abs / max(self.n, 1)) * max_speed
        rmse = ((self.sum_sq / max(self.n, 1)) ** 0.5) * max_speed
        mape = (self.sum_ape / max(self.n, 1)) * 100.0

        mean_y = self.sum_y / max(self.n, 1)
        var = max(self.sum_y2 - self.n * (mean_y ** 2), self.eps)
        rse = (self.sum_sq / (var + self.eps)) ** 0.5

        rel_sse = rse ** 2
        gain = 1.0 - rel_sse
        return {"MAE": mae, "RMSE": rmse, "MAPE": mape, "RSE": rse, "REL_SSE": rel_sse, "GAIN": gain}


@torch.no_grad()
def collect_samples(loader, device, n_samples=50):
    xs, ys = [], []
    for xb, yb in loader:
        for i in range(xb.shape[0]):
            xs.append(xb[i:i+1].to(device))
            ys.append(yb[i:i+1].to(device).squeeze(1))
            if len(xs) >= n_samples:
                return xs, ys
    return xs, ys


@torch.no_grad()
def eval_clean_on_samples(model, xs, ys, max_speed=1.0):
    model.eval()
    agg = GlobalAgg()
    for x, y in zip(xs, ys):
        pred = safe_predict(model, x)
        agg.add(pred, y)

    out = agg.finalize(max_speed=max_speed)
    out.update({
        "SHIFT_RMSE": 0.0, "DX_LINF": 0.0, "DX_NZ": 0.0,
        "mean_queries": 0.0, "mean_latency_sec": 0.0,
        "count": len(xs)
    })
    return out



def eval_1vita_on_samples(oracle, xs, ys, max_speed=1.0, beta=0.05, pop_size=15, iters=60, objective="mse_to_true", nz_thr=1e-4):
    agg = GlobalAgg()
    shift_rmse_sum = 0.0
    dx_linf_sum = 0.0
    dx_nz_sum = 0.0
    q_sum = 0.0
    q_raw_sum = 0.0
    q_sample_sum = 0.0
    t_sum = 0.0

    for j,(x,y) in enumerate(zip(xs, ys)):
        oracle.reset()
        pred_clean = oracle.predict(x)

        t0 = time.time()
        res = de_vita_attack(
            oracle=oracle, x=x, y_true=y,
            objective=objective,
            n_points=1, beta=beta, pop_size=pop_size, iters=iters,
            seed=999 + 999*j
        )
        t_sum += (time.time() - t0)

        x_adv = res["x_adv"]
        pred_adv = oracle.predict(x_adv)

        agg.add(pred_adv, y)

        shift_mse = torch.mean((pred_adv - pred_clean) ** 2).item()
        shift_rmse_sum += (shift_mse ** 0.5)

        dx = (x_adv - x).abs()
        dx_linf_sum += dx.max().item()
        dx_nz_sum += (dx > nz_thr).sum().item()

        q_sum += oracle.n_queries
        q_raw_sum += oracle.n_forward_calls
        q_sample_sum += oracle.n_sample_queries

    out = agg.finalize(max_speed=max_speed)
    n = len(xs)
    out.update({
        "SHIFT_RMSE": shift_rmse_sum / max(n,1),
        "DX_LINF": dx_linf_sum / max(n,1),
        "DX_NZ": dx_nz_sum / max(n,1),
        "mean_queries": q_sum / max(n,1),
        "mean_queries_raw": q_raw_sum / max(n,1),
        "mean_queries_sample_eq": q_sample_sum / max(n,1),
        "mean_latency_sec": t_sum / max(n,1),
        "count": n
    })
    return out

@torch.no_grad()
def eval_lba_sign_on_samples(
    oracle, flearn, xs, ys,
    max_speed=1.0,
    poison_frac=0.10,
    beta=0.06,
    delta=2.0,
    iters=8,
    restrict_last_steps=2,
    nz_thr=1e-4
):
    agg = GlobalAgg()
    shift_rmse_sum = 0.0
    dx_linf_sum = 0.0
    dx_nz_sum = 0.0
    q_sum = 0.0
    q_raw_sum = 0.0
    q_sample_sum = 0.0
    t_sum = 0.0

    for x, y in zip(xs, ys):
        oracle.reset()
        pred_clean = oracle.predict(x)

        t0 = time.time()
        x_adv = lba_sign_iterative_poison(
            oracle=oracle, flearn=flearn, x=x,
            poison_frac=poison_frac,
            beta=beta, delta=delta,
            iters=iters,
            restrict_last_steps=restrict_last_steps
        )
        t_sum += (time.time() - t0)

        pred_adv = oracle.predict(x_adv)
        agg.add(pred_adv, y)

        shift_mse = torch.mean((pred_adv - pred_clean) ** 2).item()
        shift_rmse_sum += (shift_mse ** 0.5)

        dx = (x_adv - x).abs()
        dx_linf_sum += dx.max().item()
        dx_nz_sum += (dx > nz_thr).sum().item()

        q_sum += oracle.n_queries
        q_raw_sum += oracle.n_forward_calls
        q_sample_sum += oracle.n_sample_queries

    out = agg.finalize(max_speed=max_speed)
    n = len(xs)
    out.update({
        "SHIFT_RMSE": shift_rmse_sum / max(n,1),
        "DX_LINF": dx_linf_sum / max(n,1),
        "DX_NZ": dx_nz_sum / max(n,1),
        "mean_queries": q_sum / max(n,1),
        "mean_queries_raw": q_raw_sum / max(n,1),
        "mean_queries_sample_eq": q_sample_sum / max(n,1),
        "mean_latency_sec": t_sum / max(n,1),
        "count": n
    })
    return out




def print_report(clean, vita, lba):
    def show(name, d):
        keys = ["MAE","RMSE","MAPE","RSE","REL_SSE","GAIN","SHIFT_RMSE","DX_LINF","DX_NZ","mean_queries","mean_queries_sample_eq","mean_latency_sec","count"]
        s = ", ".join([f"{k}={d[k]:.4f}" if k not in ["mean_queries","count"] else f"{k}={d[k]:.1f}" if k=="mean_queries" else f"{k}={int(d[k])}" for k in keys if k in d])
        print(f"{name:6}: {s}")
    print("\n========== ATTACK REPORT (same samples) ==========")
    show("CLEAN", clean)
    show("1VITA", vita)
    show("LBA_S", lba)
    print("===============================================\n")

In [ ]:
DADV_SAMPLES = 96
DADV_POP_SIZE = 8
DADV_ITERS = 12
FLEARN_EPOCHS = 10
EVAL_SAMPLES = 50
VITA_POP_SIZE = 10
VITA_ITERS = 30


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gclstm = gclstm.to(device)

oracle = BlackBoxOracle(gclstm, device=device)

# 1) сбор Dadv для CNN
dadv = build_dadv_dataset(
    oracle=oracle,
    loader=test_dataloader,
    n_samples=DADV_SAMPLES,
    beta=0.05,
    pop_size=DADV_POP_SIZE,
    iters=DADV_ITERS,
    objective="mse_to_clean",
    n_points=2,
)
print("Dadv size:", len(dadv))

# 2) обучение flearn
flearn = FlearnCNN2D().to(device)
flearn = train_flearn(
    flearn, dadv, device=device, lr=5e-3, batch_size=8,
    epochs=FLEARN_EPOCHS, lambda_p=1.0, topk_metric=15
)

# 3) выбор примеров
xs, ys = collect_samples(test_dataloader, device=device, n_samples=EVAL_SAMPLES)

clean = eval_clean_on_samples(gclstm, xs, ys, max_speed=max_speed)
vita = eval_1vita_on_samples(
    oracle, xs, ys, max_speed=max_speed,
    beta=0.08, pop_size=VITA_POP_SIZE, iters=VITA_ITERS,
    objective="mse_to_true"
)
lba_s = eval_lba_sign_on_samples(
    oracle, flearn, xs, ys,
    max_speed=max_speed,
    poison_frac=0.20,
    beta=0.03,
    delta=1.7,
    iters=6,
    restrict_last_steps=2
)


print_report(clean, vita, lba_s)

def save_dict_report(path, rows):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write("| Method | MAE | RMSE | MAPE | RSE | REL_SSE | GAIN | SHIFT_RMSE | DX_LINF | DX_NZ | mean_queries | mean_queries_sample_eq | mean_latency_sec |\n")
        f.write("|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|\n")
        for name, d in rows:
            f.write(
                f"| {name} | {d['MAE']:.4f} | {d['RMSE']:.4f} | {d['MAPE']:.4f} | {d['RSE']:.4f} | {d['REL_SSE']:.4f} | {d['GAIN']:.4f} | {d['SHIFT_RMSE']:.4f} | {d['DX_LINF']:.4f} | {d['DX_NZ']:.1f} | {d.get('mean_queries', 0.0):.1f} | {d.get('mean_queries_sample_eq', 0.0):.1f} | {d['mean_latency_sec']:.4f} |\n"
            )

save_dict_report(
    os.path.join("results", "final_tables.md"),
    [("CLEAN", clean), ("1VITA", vita), ("LBA_S", lba_s)]
)

Dadv size: 96
[flearn] epoch=001 loss=6.9304 AR@1=0.000 AR@15=0.073 p_RMSE@trueLoc=0.5400 lr=0.005000
[flearn] epoch=002 loss=6.2258 AR@1=0.042 AR@15=0.156 p_RMSE@trueLoc=0.1725 lr=0.004878
[flearn] epoch=003 loss=6.1611 AR@1=0.010 AR@15=0.135 p_RMSE@trueLoc=0.1151 lr=0.004523
[flearn] epoch=004 loss=6.0388 AR@1=0.000 AR@15=0.177 p_RMSE@trueLoc=0.1225 lr=0.003969
[flearn] epoch=005 loss=6.0282 AR@1=0.010 AR@15=0.177 p_RMSE@trueLoc=0.1016 lr=0.003273
[flearn] epoch=006 loss=5.9598 AR@1=0.021 AR@15=0.167 p_RMSE@trueLoc=0.0746 lr=0.002500
[flearn] epoch=007 loss=5.9065 AR@1=0.010 AR@15=0.177 p_RMSE@trueLoc=0.0712 lr=0.001727
[flearn] epoch=008 loss=5.8372 AR@1=0.021 AR@15=0.229 p_RMSE@trueLoc=0.0673 lr=0.001031
[flearn] epoch=009 loss=5.7798 AR@1=0.031 AR@15=0.219 p_RMSE@trueLoc=0.0605 lr=0.000477
[flearn] epoch=010 loss=5.7796 AR@1=0.031 AR@15=0.229 p_RMSE@trueLoc=0.0509 lr=0.000122

========== ATTACK REPORT (same samples) ==========
CLEAN : MAE=2.8699, RMSE=4.2018, MAPE=7.0519, RSE=0.33

In [ ]:
# Dadv size: 96
# [flearn] epoch=001 loss=6.9304 AR@1=0.000 AR@15=0.073 p_RMSE@trueLoc=0.5400 lr=0.005000
# [flearn] epoch=002 loss=6.2258 AR@1=0.042 AR@15=0.156 p_RMSE@trueLoc=0.1725 lr=0.004878
# [flearn] epoch=003 loss=6.1611 AR@1=0.010 AR@15=0.135 p_RMSE@trueLoc=0.1151 lr=0.004523
# [flearn] epoch=004 loss=6.0388 AR@1=0.000 AR@15=0.177 p_RMSE@trueLoc=0.1225 lr=0.003969
# [flearn] epoch=005 loss=6.0282 AR@1=0.010 AR@15=0.177 p_RMSE@trueLoc=0.1016 lr=0.003273
# [flearn] epoch=006 loss=5.9598 AR@1=0.021 AR@15=0.167 p_RMSE@trueLoc=0.0746 lr=0.002500
# [flearn] epoch=007 loss=5.9065 AR@1=0.010 AR@15=0.177 p_RMSE@trueLoc=0.0712 lr=0.001727
# [flearn] epoch=008 loss=5.8372 AR@1=0.021 AR@15=0.229 p_RMSE@trueLoc=0.0673 lr=0.001031
# [flearn] epoch=009 loss=5.7798 AR@1=0.031 AR@15=0.219 p_RMSE@trueLoc=0.0605 lr=0.000477
# [flearn] epoch=010 loss=5.7796 AR@1=0.031 AR@15=0.229 p_RMSE@trueLoc=0.0509 lr=0.000122

# ========== ATTACK REPORT (same samples) ==========
# CLEAN : MAE=2.8699, RMSE=4.2018, MAPE=7.0519, RSE=0.3349, REL_SSE=0.1121, GAIN=0.8879, SHIFT_RMSE=0.0000, DX_LINF=0.0000, DX_NZ=0.0000, mean_queries=0.0, mean_latency_sec=0.0000, count=50
# 1VITA : MAE=2.8823, RMSE=4.2284, MAPE=7.1013, RSE=0.3370, REL_SSE=0.1136, GAIN=0.8864, SHIFT_RMSE=0.0009, DX_LINF=0.0280, DX_NZ=1.0000, mean_queries=314.0, mean_queries_sample_eq=314.0000, mean_latency_sec=2.6985, count=50
# LBA_S : MAE=15.6818, RMSE=17.0757, MAPE=31.1117, RSE=1.3609, REL_SSE=1.8521, GAIN=-0.8521, SHIFT_RMSE=0.1013, DX_LINF=0.1076, DX_NZ=646.0000, mean_queries=15.0, mean_queries_sample_eq=15.0000, mean_latency_sec=0.1009, count=50
# ===============================================

In [ ]:
# [Dadv] collected 100/300, last queries=130, last latency=0.92s
# [Dadv] collected 200/300, last queries=130, last latency=0.92s
# [Dadv] collected 300/300, last queries=130, last latency=0.99s
# Dadv size: 300
# [flearn] epoch=001 loss=6.1241 AR@1=0.010 AR@15=0.150 p_RMSE@trueLoc=0.1048
# [flearn] epoch=002 loss=5.7399 AR@1=0.033 AR@15=0.217 p_RMSE@trueLoc=0.0962
# [flearn] epoch=003 loss=5.6388 AR@1=0.037 AR@15=0.240 p_RMSE@trueLoc=0.0824
# [flearn] epoch=004 loss=5.5515 AR@1=0.043 AR@15=0.283 p_RMSE@trueLoc=0.0739
# [flearn] epoch=005 loss=5.5409 AR@1=0.057 AR@15=0.283 p_RMSE@trueLoc=0.0345
# [flearn] epoch=006 loss=5.6112 AR@1=0.043 AR@15=0.253 p_RMSE@trueLoc=0.0481
# [flearn] epoch=007 loss=5.5226 AR@1=0.053 AR@15=0.277 p_RMSE@trueLoc=0.0414
# [flearn] epoch=008 loss=5.5032 AR@1=0.053 AR@15=0.277 p_RMSE@trueLoc=0.0380
# [flearn] epoch=009 loss=5.5159 AR@1=0.047 AR@15=0.273 p_RMSE@trueLoc=0.0516
# [flearn] epoch=010 loss=5.4834 AR@1=0.047 AR@15=0.277 p_RMSE@trueLoc=0.0302
# [flearn] epoch=011 loss=5.4940 AR@1=0.053 AR@15=0.287 p_RMSE@trueLoc=0.0263
# [flearn] epoch=012 loss=5.5025 AR@1=0.050 AR@15=0.273 p_RMSE@trueLoc=0.0293
# [flearn] epoch=013 loss=5.5072 AR@1=0.050 AR@15=0.270 p_RMSE@trueLoc=0.0298
# [flearn] epoch=014 loss=5.5215 AR@1=0.043 AR@15=0.270 p_RMSE@trueLoc=0.0239
# [flearn] epoch=015 loss=5.4388 AR@1=0.070 AR@15=0.277 p_RMSE@trueLoc=0.0301
# [flearn] epoch=016 loss=5.4574 AR@1=0.057 AR@15=0.287 p_RMSE@trueLoc=0.0219
# [flearn] epoch=017 loss=5.4350 AR@1=0.063 AR@15=0.280 p_RMSE@trueLoc=0.0401
# [flearn] epoch=018 loss=5.4580 AR@1=0.060 AR@15=0.273 p_RMSE@trueLoc=0.0389
# [flearn] epoch=019 loss=5.4511 AR@1=0.070 AR@15=0.283 p_RMSE@trueLoc=0.0378
# [flearn] epoch=020 loss=5.4051 AR@1=0.070 AR@15=0.293 p_RMSE@trueLoc=0.0432
# [flearn] epoch=021 loss=5.4168 AR@1=0.063 AR@15=0.307 p_RMSE@trueLoc=0.0333
# [flearn] epoch=022 loss=5.4548 AR@1=0.053 AR@15=0.287 p_RMSE@trueLoc=0.0496
# [flearn] epoch=023 loss=5.3998 AR@1=0.070 AR@15=0.307 p_RMSE@trueLoc=0.0494
# [flearn] epoch=024 loss=5.3823 AR@1=0.067 AR@15=0.293 p_RMSE@trueLoc=0.0249
# [flearn] epoch=025 loss=5.4349 AR@1=0.073 AR@15=0.303 p_RMSE@trueLoc=0.0355
# [flearn] epoch=026 loss=5.4033 AR@1=0.080 AR@15=0.303 p_RMSE@trueLoc=0.0279
# [flearn] epoch=027 loss=5.3956 AR@1=0.063 AR@15=0.297 p_RMSE@trueLoc=0.0276
# [flearn] epoch=028 loss=5.3812 AR@1=0.073 AR@15=0.290 p_RMSE@trueLoc=0.0342
# [flearn] epoch=029 loss=5.3895 AR@1=0.057 AR@15=0.313 p_RMSE@trueLoc=0.0285
# [flearn] epoch=030 loss=5.3840 AR@1=0.063 AR@15=0.317 p_RMSE@trueLoc=0.0254

# ========== ATTACK REPORT (same samples) ==========
# CLEAN : MAE=2.7370, RMSE=4.0378, MAPE=5.9100, RSE=0.4163, REL_SSE=0.1733, GAIN=0.8267, SHIFT_RMSE=0.0000, DX_LINF=0.0000, DX_NZ=0.0000, mean_queries=0.0, mean_latency_sec=0.0000, count=50
# 1VITA : MAE=2.7479, RMSE=4.0610, MAPE=5.9423, RSE=0.4187, REL_SSE=0.1753, GAIN=0.8247, SHIFT_RMSE=0.0007, DX_LINF=0.0270, DX_NZ=1.0000, mean_queries=314.0, mean_latency_sec=2.5645, count=50
# LBA_S : MAE=7.4977, RMSE=8.5819, MAPE=14.0032, RSE=0.8848, REL_SSE=0.7829, GAIN=0.2171, SHIFT_RMSE=0.0467, DX_LINF=0.0514, DX_NZ=808.0000, mean_queries=13.0, mean_latency_sec=0.0852, count=50
# ===============================================


In [ ]:
# [Dadv] collected 100/300, last queries=130, last latency=0.87s
# [Dadv] collected 200/300, last queries=130, last latency=0.91s
# [Dadv] collected 300/300, last queries=130, last latency=0.92s
# Dadv size: 300
# [flearn] epoch=001 loss=6.2953 AR@1=0.013 AR@15=0.197 p_RMSE@trueLoc=0.2106
# [flearn] epoch=002 loss=5.7184 AR@1=0.010 AR@15=0.197 p_RMSE@trueLoc=0.0727
# [flearn] epoch=003 loss=5.6335 AR@1=0.017 AR@15=0.220 p_RMSE@trueLoc=0.0475
# [flearn] epoch=004 loss=5.6391 AR@1=0.017 AR@15=0.247 p_RMSE@trueLoc=0.0497
# [flearn] epoch=005 loss=5.5548 AR@1=0.020 AR@15=0.250 p_RMSE@trueLoc=0.0489
# [flearn] epoch=006 loss=5.5072 AR@1=0.030 AR@15=0.260 p_RMSE@trueLoc=0.0487
# [flearn] epoch=007 loss=5.5327 AR@1=0.033 AR@15=0.270 p_RMSE@trueLoc=0.0429
# [flearn] epoch=008 loss=5.5193 AR@1=0.030 AR@15=0.267 p_RMSE@trueLoc=0.0342
# [flearn] epoch=009 loss=5.5089 AR@1=0.037 AR@15=0.270 p_RMSE@trueLoc=0.0327
# [flearn] epoch=010 loss=5.5085 AR@1=0.033 AR@15=0.280 p_RMSE@trueLoc=0.0364
# [flearn] epoch=011 loss=5.4541 AR@1=0.040 AR@15=0.287 p_RMSE@trueLoc=0.0266
# [flearn] epoch=012 loss=5.4683 AR@1=0.027 AR@15=0.270 p_RMSE@trueLoc=0.0375
# [flearn] epoch=013 loss=5.4682 AR@1=0.037 AR@15=0.307 p_RMSE@trueLoc=0.0357
# [flearn] epoch=014 loss=5.4737 AR@1=0.027 AR@15=0.290 p_RMSE@trueLoc=0.0385
# [flearn] epoch=015 loss=5.4427 AR@1=0.030 AR@15=0.277 p_RMSE@trueLoc=0.0432
# [flearn] epoch=016 loss=5.4480 AR@1=0.027 AR@15=0.277 p_RMSE@trueLoc=0.0390
# [flearn] epoch=017 loss=5.4351 AR@1=0.037 AR@15=0.287 p_RMSE@trueLoc=0.0322
# [flearn] epoch=018 loss=5.4443 AR@1=0.037 AR@15=0.280 p_RMSE@trueLoc=0.0321
# [flearn] epoch=019 loss=5.4252 AR@1=0.023 AR@15=0.267 p_RMSE@trueLoc=0.0264
# [flearn] epoch=020 loss=5.4214 AR@1=0.050 AR@15=0.290 p_RMSE@trueLoc=0.0315
# [flearn] epoch=021 loss=5.4169 AR@1=0.040 AR@15=0.277 p_RMSE@trueLoc=0.0333
# [flearn] epoch=022 loss=5.3802 AR@1=0.043 AR@15=0.280 p_RMSE@trueLoc=0.0308
# [flearn] epoch=023 loss=5.3952 AR@1=0.027 AR@15=0.273 p_RMSE@trueLoc=0.0326
# [flearn] epoch=024 loss=5.4004 AR@1=0.053 AR@15=0.267 p_RMSE@trueLoc=0.0329
# [flearn] epoch=025 loss=5.4116 AR@1=0.037 AR@15=0.277 p_RMSE@trueLoc=0.0277
# [flearn] epoch=026 loss=5.3733 AR@1=0.047 AR@15=0.303 p_RMSE@trueLoc=0.0269
# [flearn] epoch=027 loss=5.3594 AR@1=0.043 AR@15=0.300 p_RMSE@trueLoc=0.0263
# [flearn] epoch=028 loss=5.3945 AR@1=0.050 AR@15=0.283 p_RMSE@trueLoc=0.0287
# [flearn] epoch=029 loss=5.3476 AR@1=0.053 AR@15=0.297 p_RMSE@trueLoc=0.0257
# [flearn] epoch=030 loss=5.3396 AR@1=0.053 AR@15=0.300 p_RMSE@trueLoc=0.0243

# ========== ATTACK REPORT (same samples) ==========
# CLEAN : MAE=2.9581, RMSE=4.2645, MAPE=6.8899, RSE=0.3604, REL_SSE=0.1299, GAIN=0.8701, SHIFT_RMSE=0.0000, DX_LINF=0.0000, DX_NZ=0.0000, mean_queries=0.0, mean_latency_sec=0.0000, count=50
# 1VITA : MAE=2.9700, RMSE=4.2987, MAPE=6.9445, RSE=0.3633, REL_SSE=0.1320, GAIN=0.8680, SHIFT_RMSE=0.0009, DX_LINF=0.0269, DX_NZ=1.0000, mean_queries=3029.0, mean_latency_sec=24.6917, count=50
# LBA_S : MAE=4.5965, RMSE=5.7531, MAPE=9.5426, RSE=0.4862, REL_SSE=0.2364, GAIN=0.7636, SHIFT_RMSE=0.0238, DX_LINF=0.0262, DX_NZ=646.0000, mean_queries=9.0, mean_latency_sec=0.0516, count=50
# ===============================================

In [ ]:
# [Dadv] collected 100/300, last queries=130, last latency=0.89s
# [Dadv] collected 200/300, last queries=130, last latency=0.90s
# [Dadv] collected 300/300, last queries=130, last latency=0.90s
# Dadv size: 300
# [flearn] epoch=001 loss=6.2214 AR@1=0.007 AR@15=0.153 p_RMSE@trueLoc=0.1193
# [flearn] epoch=002 loss=5.8446 AR@1=0.027 AR@15=0.187 p_RMSE@trueLoc=0.0467
# [flearn] epoch=003 loss=5.8386 AR@1=0.023 AR@15=0.210 p_RMSE@trueLoc=0.0652
# [flearn] epoch=004 loss=5.7584 AR@1=0.027 AR@15=0.230 p_RMSE@trueLoc=0.0356
# [flearn] epoch=005 loss=5.7405 AR@1=0.023 AR@15=0.230 p_RMSE@trueLoc=0.0636
# [flearn] epoch=006 loss=5.6936 AR@1=0.037 AR@15=0.220 p_RMSE@trueLoc=0.0295
# [flearn] epoch=007 loss=5.6790 AR@1=0.020 AR@15=0.247 p_RMSE@trueLoc=0.0373
# [flearn] epoch=008 loss=5.7344 AR@1=0.017 AR@15=0.217 p_RMSE@trueLoc=0.0356
# [flearn] epoch=009 loss=5.7938 AR@1=0.040 AR@15=0.230 p_RMSE@trueLoc=0.0516
# [flearn] epoch=010 loss=5.6805 AR@1=0.033 AR@15=0.237 p_RMSE@trueLoc=0.0301
# [flearn] epoch=011 loss=5.6739 AR@1=0.040 AR@15=0.233 p_RMSE@trueLoc=0.0342
# [flearn] epoch=012 loss=5.6694 AR@1=0.027 AR@15=0.243 p_RMSE@trueLoc=0.0362
# [flearn] epoch=013 loss=5.6906 AR@1=0.027 AR@15=0.237 p_RMSE@trueLoc=0.0348
# [flearn] epoch=014 loss=5.6781 AR@1=0.037 AR@15=0.250 p_RMSE@trueLoc=0.0321
# [flearn] epoch=015 loss=5.6779 AR@1=0.023 AR@15=0.240 p_RMSE@trueLoc=0.0355
# [flearn] epoch=016 loss=5.6714 AR@1=0.037 AR@15=0.237 p_RMSE@trueLoc=0.0257
# [flearn] epoch=017 loss=5.6726 AR@1=0.030 AR@15=0.237 p_RMSE@trueLoc=0.0325
# [flearn] epoch=018 loss=5.6741 AR@1=0.037 AR@15=0.247 p_RMSE@trueLoc=0.0313
# [flearn] epoch=019 loss=5.6555 AR@1=0.033 AR@15=0.230 p_RMSE@trueLoc=0.0303
# [flearn] epoch=020 loss=5.6321 AR@1=0.037 AR@15=0.250 p_RMSE@trueLoc=0.0313
# [flearn] epoch=021 loss=5.6722 AR@1=0.020 AR@15=0.250 p_RMSE@trueLoc=0.0322
# [flearn] epoch=022 loss=5.6670 AR@1=0.043 AR@15=0.243 p_RMSE@trueLoc=0.0448
# [flearn] epoch=023 loss=5.6731 AR@1=0.037 AR@15=0.247 p_RMSE@trueLoc=0.0679
# [flearn] epoch=024 loss=5.6577 AR@1=0.030 AR@15=0.243 p_RMSE@trueLoc=0.0370
# [flearn] epoch=025 loss=5.6254 AR@1=0.030 AR@15=0.237 p_RMSE@trueLoc=0.0247
# [flearn] epoch=026 loss=5.5990 AR@1=0.040 AR@15=0.253 p_RMSE@trueLoc=0.0234
# [flearn] epoch=027 loss=5.6202 AR@1=0.043 AR@15=0.230 p_RMSE@trueLoc=0.0262
# [flearn] epoch=028 loss=5.6003 AR@1=0.047 AR@15=0.270 p_RMSE@trueLoc=0.0338
# [flearn] epoch=029 loss=5.6229 AR@1=0.047 AR@15=0.263 p_RMSE@trueLoc=0.0395
# [flearn] epoch=030 loss=5.5921 AR@1=0.043 AR@15=0.247 p_RMSE@trueLoc=0.0261

# ========== ATTACK REPORT (same samples) ==========
# CLEAN : MAE=2.9201, RMSE=4.2548, MAPE=7.1411, RSE=0.3523, REL_SSE=0.1241, GAIN=0.8759, SHIFT_RMSE=0.0000, DX_LINF=0.0000, DX_NZ=0.0000, mean_queries=0.0, mean_latency_sec=0.0000, count=50
# 1VITA : MAE=2.9325, RMSE=4.2814, MAPE=7.1739, RSE=0.3545, REL_SSE=0.1257, GAIN=0.8743, SHIFT_RMSE=0.0008, DX_LINF=0.0280, DX_NZ=1.0000, mean_queries=314.0, mean_latency_sec=2.5736, count=50
# LBA_S : MAE=6.4091, RMSE=7.6014, MAPE=13.4509, RSE=0.6294, REL_SSE=0.3962, GAIN=0.6038, SHIFT_RMSE=0.0390, DX_LINF=0.0422, DX_NZ=646.0000, mean_queries=11.0, mean_latency_sec=0.0677, count=50

In [ ]:
# rnn_val_loss = np.asarray(rnn_loss[3])
# lstm_val_loss = np.asarray(lstm_loss[3])
# hgclstm_val_loss = np.asarray(gclstm_loss[3])
# lsgclstm_val_loss = np.asarray(lsgclstm_loss[3])
# sgclstm_val_loss = np.asarray(sgclstm_loss[3])

In [ ]:
# lstm_val_loss = np.load('lstm_val_loss.npy')
# hgclstm_val_loss = np.load('hgclstm_val_loss.npy')
# lsgclstm_val_loss = np.load('lsgclstm_val_loss.npy')
# sgclstm_val_loss = np.load('sgclstm_val_loss.npy')

In [ ]:
# np.save('lstm_val_loss', lstm_val_loss)
# np.save('hgclstm_val_loss', gclstm_val_loss)
# np.save('lsgclstm_val_loss', lsgclstm_val_loss)
# np.save('sgclstm_val_loss', sgclstm_val_loss)

In [ ]:
# fig, ax = plt.subplots()
# plt.plot(np.arange(1, len(lstm_val_loss) + 1), lstm_val_loss,  label = 'LSTM')
# plt.plot(np.arange(1, len(sgclstm_val_loss) + 1), sgclstm_val_loss,  label = 'SGC+LSTM')
# plt.plot(np.arange(1, len(lsgclstm_val_loss) + 1), lsgclstm_val_loss,  label = 'LSGC+LSTM')
# plt.plot(np.arange(1, len(hgclstm_val_loss) + 1),hgclstm_val_loss,  label = 'HGC-LSTM')
# plt.ylim((6 * 0.0001, 0.0019))
# plt.xticks(fontsize = 12)
# plt.yticks(fontsize = 12)
# plt.yscale('log')
# plt.ylabel('Validation Loss (MSE)', fontsize=12)
# plt.xlabel('Epoch', fontsize=12)
# # plt.gca().invert_xaxis()
# plt.legend(fontsize=14)
# plt.grid(True, which='both')

# plt.savefig('Validation_loss.png', dpi=300, bbox_inches = 'tight', pad_inches=0.1)